# SI 2. GIWAXS reduction and peak analysis

Xeuss 3.0, Cu K$\alpha$ ($\lambda$ = 1.5419 Å), Dectris EIGER2 1M (75 µm pixels), sample to detector distance 72.0 mm, nominal incidence 0.20°. The frame is `ZnPC_Xuess GIWAXS 1.edf` (header title `ZnPC_Harlan_0_00000`, 2024-03-18, 600 s), the ZnPc film on native-oxide Si on which P-RSoXR was measured, and the frame behind the NIKA figure in `ZnPc_GIWAXS_graph.pxp`. It is reduced here from the raw EDF image with no NIKA step.

The notebook covers the detector to reciprocal-space transform and its forbidden wedge, a check of the beam centre and incidence angle, validation of the $q$ scale against the NIKA sector cuts of the same frame, sector cuts with pseudo-Voigt fits, the reflections of the β and α cells in fiber texture, the orientation distribution of the lamellar ring, and the figure.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import gemmi
import matplotlib
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, LogLocator, MultipleLocator, NullFormatter
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

from utils.helpers.plotting_helper import set_plotting_defaults

import giwaxs_style as gws   # shared with FIG4 (SI 12), lives beside this notebook

set_plotting_defaults()

ROOT = Path.home() / "projects/refl-analysis"
OUT_DIR = ROOT / "notebooks/manuscript"
CELLS = {
    "β (CIF, 100 K)": ROOT / "md/znpc_film/ff/tpZnPc_0m.cif",
    "α constrained": ROOT / "md/znpc_film/cell/unit_cells/alpha_ZnPc_GIXRD_constrained.cif",
    "α relaxed (MD)": ROOT / "md/znpc_film/cell/unit_cells/alpha_ZnPc_MD_relaxed.cif",
}
NIKA_CSV = OUT_DIR / "XuessGiwaxs_ZnPC_Harlan.csv"   # NIKA 90_10 and 170_10 sectors (smoothed), exported from ZnPc_GIWAXS_graph.pxp

# direct-beam centres (pixel x, pixel y) from the transmitted spot, see the beam-geometry section
FRAMES = {"ZnPC_Xuess GIWAXS 1.edf": (584.0, 982.0)}     # same centre as the NIKA reduction
PRIMARY = "ZnPC_Xuess GIWAXS 1.edf"

CHI_OOP = 25.0      # figure sector, out of plane, chi < CHI_OOP
CHI_IP = 73.0       # figure sector, in plane, chi > CHI_IP on the q_xy < 0 side
MERIDIAN = 5.0      # fit sector, chi < chi_edge(q) + MERIDIAN
RHO_FILM = 1.6      # g/cm3, for the critical angle only
K_SCHERRER = 0.9

## Detector to reciprocal space

A pixel at column $X$ and row $Y$ lies at $\Delta x = (X - x_0)\,p$ and $\Delta z = (y_0 - Y)\,p$ from the direct beam $(x_0, y_0)$, with $p$ = 75 µm and $L$ the sample to detector distance. The in-plane exit angle is $2\theta_f = \arctan(\Delta x / L)$ and the exit angle above the sample horizon is $\alpha_f = \arctan\!\big(\Delta z / \sqrt{L^2 + \Delta x^2}\big) - \alpha_i$. With $k = 2\pi/\lambda$ the scattering vector in the sample frame is

$q_x = k(\cos\alpha_f \cos 2\theta_f - \cos\alpha_i)$, $q_y = k \cos\alpha_f \sin 2\theta_f$, $q_z = k(\sin\alpha_f + \sin\alpha_i)$,

and the map is drawn against $q_{xy} = \pm\sqrt{q_x^2 + q_y^2}$, signed by the detector side. The polar angle of $\mathbf{q}$ from the film normal is $\chi = \arctan(|q_{xy}|/q_z)$. Pixels below the horizon ($\alpha_f < 0$) and masked pixels (negative counts) are dropped.

The forbidden wedge follows from the transform. On the detector meridian ($2\theta_f$ = 0), $q_{xy} = k(\cos\alpha_i - \cos\alpha_f)$ is not zero, so the smallest accessible $\chi$ at a given $|q|$ is $\chi_\mathrm{edge}(q)$, about 3.3°, 6.8° and 10.4° at $q$ = 0.5, 1.0 and 1.5 Å$^{-1}$. Bins with no contributing pixel are left empty in the map and never interpolated. Sector cuts on $|q|$ therefore sample a $q$-dependent range of $\chi$, and a sector defined by $\chi < \chi_0$ contains no pixels where $\chi_\mathrm{edge}(q) > \chi_0$. The lamellar fits use a meridian sector that follows the wedge edge, $\chi < \chi_\mathrm{edge}(q) + 5°$, which keeps the off-specular ($h0l$) reflections near $\chi \approx 25$ to 30° out of the cut at every order.

In [ ]:
def read_edf(path):
    """ESRF data format, one image. Header is ASCII key = value ; pairs inside braces, data follow as little-endian float32."""
    raw = Path(path).read_bytes()
    end = raw.find(b"}") + 1
    fields = (f.split("=", 1) for f in raw[:end].decode().strip("{}\n").split(";") if "=" in f)
    hdr = {k.strip(): v.strip() for k, v in fields}
    n = int(hdr["EDF_BinarySize"])
    img = np.frombuffer(raw[-n:], "<f4").reshape(int(hdr["Dim_2"]), int(hdr["Dim_1"])).astype(float)
    return img, hdr


def gi_transform(img, hdr, cx, cy, ai_deg=None, L=None):
    L = float(hdr["SampleDistance"]) if L is None else L
    p = float(hdr["PSize_1"])
    k = 2 * np.pi / (float(hdr["WaveLength"]) * 1e10)
    ai = np.radians(float(hdr["om"]) if ai_deg is None else ai_deg)
    ny, nx = img.shape
    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))
    dx, dz = (X - cx) * p, (cy - Y) * p
    tth = np.arctan2(dx, L)
    af = np.arctan2(dz, np.hypot(L, dx)) - ai
    qx = k * (np.cos(af) * np.cos(tth) - np.cos(ai))
    qy = k * np.cos(af) * np.sin(tth)
    qz = k * (np.sin(af) + np.sin(ai))
    qxy = np.where(dx < 0, -1.0, 1.0) * np.hypot(qx, qy)
    q = np.sqrt(qx**2 + qy**2 + qz**2)
    chi = np.degrees(np.arctan2(np.abs(qxy), qz))
    ok = np.isfinite(img) & (img > 0) & (af >= 0)
    return dict(qxy=qxy, qz=qz, q=q, chi=chi, af=af, ok=ok, k=k, ai=ai, L=L, p=p)


def chi_edge(q, k, ai):
    af = np.linspace(0, np.radians(45), 6000)
    qxy, qz = k * (np.cos(ai) - np.cos(af)), k * (np.sin(af) + np.sin(ai))
    return np.interp(q, np.hypot(qxy, qz), np.degrees(np.arctan2(qxy, qz)))


def qmap(img, g, qxy_lim=(-2.15, 0.62), qz_lim=(0.0, 1.45), dq=0.008):
    bx = np.arange(qxy_lim[0], qxy_lim[1] + dq, dq)
    bz = np.arange(qz_lim[0], qz_lim[1] + dq, dq)
    v = g["ok"]
    S, _, _ = np.histogram2d(g["qxy"][v], g["qz"][v], [bx, bz], weights=img[v])
    N, _, _ = np.histogram2d(g["qxy"][v], g["qz"][v], [bx, bz])
    return bx, bz, np.where(N > 0, S / np.maximum(N, 1), np.nan).T


def sector(img, g, sel, qb):
    s, _ = np.histogram(g["q"][sel], qb, weights=img[sel])
    n, _ = np.histogram(g["q"][sel], qb)
    return 0.5 * (qb[1:] + qb[:-1]), np.where(n > 0, s / np.maximum(n, 1), np.nan)


def sectors(img, g):
    ce = chi_edge(g["q"], g["k"], g["ai"])
    return {
        "oop": g["ok"] & (g["chi"] <= CHI_OOP),
        "ip": g["ok"] & (g["chi"] >= CHI_IP) & (g["qxy"] < 0),
        "meridian": g["ok"] & (g["chi"] < ce + MERIDIAN),
    }


DATA = {}
for f, (cx, cy) in FRAMES.items():
    img, hdr = read_edf(ROOT / f)
    g = gi_transform(img, hdr, cx, cy)
    DATA[f] = dict(img=img, hdr=hdr, g=g, sel=sectors(img, g), cx=cx, cy=cy)
    print(f"{f:26s} {hdr['Date']}  {hdr['ExposureTime']:>5s} s  L = {g['L']*1e3:.2f} mm  alpha_i = {np.degrees(g['ai']):.4f} deg")
print("chi_edge at q = 0.5, 1.0, 1.5 1/A:", chi_edge(np.array([0.5, 1.0, 1.5]), g["k"], g["ai"]).round(1))

## Beam centre, horizon and incidence angle

The EDF header centre (607.1, 1000.0) is not the direct beam for this frame. The direct beam is taken as the intensity centroid of the transmitted spot below the sample. The specular spot sits above it at $L\tan 2\alpha_i$, and the sample horizon (the shadow edge, with the Yoneda maximum at $\alpha_f = \alpha_c$ just above it) at $L\tan\alpha_i$. At $L$ = 72 mm one pixel subtends 0.060°, so the separations below measure $\alpha_i$ only to about ±0.03° per pixel of centroid error. The Yoneda row drifts by a few pixels across the detector, a sample or detector roll of about 0.3°, which rotates the map by the same angle. It changes $|q|$ by less than 10$^{-4}$ relative and $\chi$ by 0.3°, and is not corrected.

In [ ]:
def centroid(profile, rows, lo, hi):
    m = (rows >= lo) & (rows <= hi)
    return np.sum(profile[m] * rows[m]) / np.sum(profile[m])


GEOM = {}
for f, D in DATA.items():
    img, g, cx = D["img"], D["g"], int(round(D["cx"]))
    rows = np.arange(950, 1000)
    col = img[950:1000, cx - 6:cx + 7].sum(1)
    y_dir = centroid(col, rows, D["cy"] - 2.5, D["cy"] + 2.5)
    y_spec = centroid(col, rows, y_dir - 8.5, y_dir - 3.5)
    xs = np.r_[np.arange(100, 520, 20), np.arange(660, 1000, 20)]
    yon = []
    for x in xs:
        pr = img[940:1000, x:x + 20].copy()
        pr[pr <= 0] = np.nan
        prof = np.nanmedian(pr, 1) if np.isfinite(pr).any(axis=1).all() else np.nan_to_num(np.nanmean(np.nan_to_num(pr), 1))
        yon.append(940 + int(np.argmax(np.nan_to_num(prof))))
    slope, icpt = np.polyfit(xs, yon, 1)
    y_yon = slope * D["cx"] + icpt
    px = lambda n: np.degrees(np.arctan(n * g["p"] / g["L"]))
    GEOM[f] = dict(y_direct=y_dir, y_specular=y_spec, y_yoneda=y_yon, roll_deg=np.degrees(np.arctan(slope)),
                   ai_specular=px(y_dir - y_spec) / 2, ai_plus_ac_yoneda=px(y_dir - y_yon))
pd.DataFrame(GEOM).T.round(3)

The specular separation gives $\alpha_i \approx$ 0.16°, below the header value of 0.20°, and the specular and Yoneda rows nearly coincide, which is the signature of $\alpha_i \approx \alpha_c$. The film critical angle for $\rho$ = 1.6 g cm$^{-3}$ is computed below. The measurement therefore sits at or just above the film critical angle and below that of Si (0.22°). Two consequences follow. The probed depth is set by the evanescent or weakly transmitted wave and may weight the vacuum side of the film, and the refraction correction to $q_z$ is large and poorly conditioned. The reduction keeps the header $\alpha_i$ and reports uncorrected (vacuum) $q$, the convention of the NIKA export, and carries the refraction correction as a systematic bound on the out-of-plane positions.

In [ ]:
def critical_angle(rho, lam=1.5419):
    z_e, mw = 32 * 6 + 16 + 8 * 7 + 30, 577.9            # electrons and molar mass of C32H16N8Zn
    rho_e = rho / mw * 6.02214e23 * z_e * 1e-24          # e / A^3
    delta = 2.8179e-5 * lam**2 * rho_e / (2 * np.pi)
    return np.degrees(np.sqrt(2 * delta))


def qz_refraction(qz, k, ai, ac):
    """Film-frame q_z from the vacuum q_z, both beams refracted at the vacuum-film interface."""
    sa = qz / k - np.sin(ai)
    kin = np.sqrt(np.maximum(np.sin(ai) ** 2 - np.sin(ac) ** 2, 0))
    kout = np.sqrt(np.maximum(sa**2 - np.sin(ac) ** 2, 0))
    return k * (kin + kout)


AC = critical_angle(RHO_FILM)
g = DATA[PRIMARY]["g"]
print(f"alpha_c(ZnPc, {RHO_FILM} g/cm3) = {AC:.3f} deg")
for ai_deg in (0.20, 0.19):
    qz_c = qz_refraction(0.500, g["k"], np.radians(ai_deg), np.radians(AC))
    print(f"alpha_i = {ai_deg:.2f} deg: q_z 0.500 -> {qz_c:.4f} 1/A (d {2*np.pi/0.5:.2f} -> {2*np.pi/qz_c:.2f} A)")

## Validation of the $q$ scale against NIKA

The NIKA reduction of this frame (beam centre 584, 982, distance 72 mm, sectors 90 ± 10° and 170 ± 10° in detector azimuth) is fitted with the same pseudo-Voigt model as the cuts here. The NIKA curves were exported smoothed, which broadens them slightly but leaves the centres. The 90° detector-azimuth sector is close to the wedge-following meridian sector, and the 170° sector to the in-plane sector. NIKA uses $|q|$ from the total scattering angle without the grazing-incidence decomposition, which agrees with $|q|$ here to first order in $\alpha_i$.

In [ ]:
def pv(x, A, x0, w, eta):
    u = (x - x0) / w
    return A * (eta / (1 + 4 * u**2) + (1 - eta) * np.exp(-4 * np.log(2) * u**2))


def fit_pv(x, y, lo, hi, x0s, w0=0.06):
    """Sum of pseudo-Voigts on a linear background in a window. Returns per-peak (q, err, fwhm, err, eta) and the model."""
    m = (x > lo) & (x < hi) & np.isfinite(y)
    xx, yy = x[m], y[m]
    n = len(x0s)
    def f(x, b0, b1, *p):
        return b0 + b1 * (x - lo) + sum(pv(x, *p[4 * i:4 * i + 4]) for i in range(n))
    p0, lb, ub = [np.min(yy), 0.0], [-np.inf, -np.inf], [np.inf, np.inf]
    for x0 in x0s:
        p0 += [max(np.interp(x0, xx, yy) - np.min(yy), 1e-6), x0, w0, 0.5]
        lb += [0, x0 - 0.08, 0.005, 0]
        ub += [np.inf, x0 + 0.08, 0.6, 1]
    P, C = curve_fit(f, xx, yy, p0=p0, bounds=(lb, ub), maxfev=40000)
    E = np.sqrt(np.diag(C))
    peaks = [dict(q=P[3 + 4 * i], dq=E[3 + 4 * i], fwhm=P[4 + 4 * i], dfwhm=E[4 + 4 * i], eta=P[5 + 4 * i]) for i in range(n)]
    return peaks, (lambda x: f(x, *P)), (lo, hi)


nika = pd.read_csv(NIKA_CSV)
qb = np.arange(0.2, 2.3, 0.004)
D = DATA[PRIMARY]
x_m, y_m = sector(D["img"], D["g"], D["sel"]["meridian"], qb)
x_i, y_i = sector(D["img"], D["g"], D["sel"]["ip"], qb)
check = []
for lab, (xh, yh), (xn, yn), win in [
    ("meridian, 1st order", (x_m, y_m), (nika.q_90_10, nika.I_90_10), (0.38, 0.66, [0.50])),
    ("meridian, 1.0 to 1.2 blend", (x_m, y_m), (nika.q_90_10, nika.I_90_10), (0.80, 1.30, [1.05])),
    ("in plane, ring", (x_i, y_i), (nika.q_170_10, nika.I_170_10), (0.38, 0.66, [0.51])),
    ("in plane, pi-pi", (x_i, y_i), (nika.q_170_10, nika.I_170_10), (1.45, 2.25, [1.89])),
]:
    a, b = fit_pv(xh, yh, *win)[0][0], fit_pv(np.asarray(xn), np.asarray(yn), *win)[0][0]
    check.append(dict(peak=lab, q_here=a["q"], q_NIKA=b["q"], rel_diff=(a["q"] - b["q"]) / b["q"], fwhm_here=a["fwhm"], fwhm_NIKA=b["fwhm"]))
pd.DataFrame(check).round(4)

## Sector cuts and peak fits

Pseudo-Voigt plus linear background in each window. The lamellar orders come from the meridian sector and the in-plane peaks from the 73 to 90° sector. In the in-plane sector the 0.50 to 0.53 Å$^{-1}$ ring is a blend of the tail of the (200) mosaic and (002), so it is fitted both as one peak and as two. Domain sizes are Scherrer lengths $L = 2\pi K/\Delta q$ with $K$ = 0.9. They are lower bounds, because the instrumental width, dominated at this distance by the beam footprint along the sample, is not deconvolved. The sample length is not in the header, so that width is not estimated here.

In [ ]:
WINDOWS = {
    "(200)": ("meridian", 0.38, 0.66, [0.50]),
    "(400), one peak": ("meridian", 0.80, 1.30, [1.05]),
    "(400) and (402)": ("meridian", 0.80, 1.35, [1.01, 1.14]),
    "(600)": ("meridian", 1.25, 1.80, [1.50]),
    "IP ring, one peak": ("ip", 0.38, 0.66, [0.51]),
    "IP ring, two peaks": ("ip", 0.38, 0.66, [0.50, 0.535]),
    "π-π": ("ip", 1.45, 2.25, [1.89]),
}
CUTS, FITS, rows = {}, {}, []
for f, D in DATA.items():
    CUTS[f] = {s: sector(D["img"], D["g"], D["sel"][s], qb) for s in ("oop", "ip", "meridian")}
    for lab, (s, lo, hi, x0) in WINDOWS.items():
        peaks, model, win = fit_pv(*CUTS[f][s], lo, hi, x0)
        FITS[(f, lab)] = (s, model, win, peaks)
        for j, pk in enumerate(peaks):
            rows.append(dict(frame=f, peak=lab + (f" [{j+1}]" if len(peaks) > 1 else ""), sector=s, q=pk["q"], q_err=pk["dq"],
                             d=2 * np.pi / pk["q"], fwhm=pk["fwhm"], fwhm_err=pk["dfwhm"], eta=pk["eta"],
                             L_scherrer=2 * np.pi * K_SCHERRER / pk["fwhm"]))
PEAKS = pd.DataFrame(rows)
PEAKS.round(4)

## Lamellar order and paracrystalline disorder

For a one-dimensional paracrystal the integral breadth of order $m$ grows as $\Delta q_m = 2\pi/L + \pi^2 g^2 q_1 m^2$ (Hosemann), with $g$ the relative spread of the lamellar spacing. The estimate is only meaningful if the meridian peaks are harmonics, so the order ratios are reported first. Fitted as one peak, the 1.0 to 1.2 Å$^{-1}$ feature sits at 2.12 $q_1$, and its centre moves from 1.07 to 1.10 Å$^{-1}$ as the meridian sector widens from 3 to 20° above the wedge edge. It is a blend of (400) and the mosaic-broadened (402)/(20$\bar{4}$) arc ($|q| \approx$ 1.14 to 1.17 Å$^{-1}$ at $\chi \approx$ 25 to 30° in the α cells), so the paracrystal analysis takes (400) from the two-peak fit. The third-order window holds only a weak hump and is reported for completeness. A linear-in-$q$ footprint width would also be absorbed partly into $g$.

In [ ]:
PARA = {}
for f in DATA:
    pk = PEAKS[(PEAKS.frame == f) & PEAKS.peak.isin(["(200)", "(400) and (402) [1]", "(600)"])].reset_index(drop=True)
    m = np.arange(1, len(pk) + 1)
    q1 = pk.q[0]
    out = dict(q1=q1, d1=2 * np.pi / q1, ratio_2=pk.q[1] / q1, ratio_3=pk.q[2] / q1)
    for nord in (2, 3):
        b, a = np.polyfit(m[:nord] ** 2, pk.fwhm[:nord], 1)
        out[f"g_{nord}orders"] = np.sqrt(max(b, 0) / (np.pi**2 * q1))
        out[f"L_{nord}orders"] = 2 * np.pi / a if a > 0 else np.inf
    PARA[f] = out
pd.DataFrame(PARA).T.round(4)

## Off-axis Bragg spots and indexing in fiber texture

The only localized off-axis feature in this frame is (002) on the Yoneda band at $|q_{xy}| \approx$ 0.52 to 0.53 Å$^{-1}$. It is fitted in the pixel cloud (not the regridded map) with a 2D Gaussian on a planar background, weighted by Poisson errors, on both $q_{xy}$ signs. The mean of the two signs cancels a horizontal centre error to first order. The other features are rings or arcs and are characterized by the sector fits above.

Predicted positions use fiber texture with the reciprocal vector of the lamellar planes along $q_z$ (for α, $\mathbf{a}^*$, the film normal of the MD frame), $q_z = \mathbf{G}\cdot\hat{\mathbf{n}}$, $q_{xy} = \sqrt{|\mathbf{G}|^2 - q_z^2}$, with $|F_{hkl}|^2$ from the CIF (gemmi, X-ray form factors) summed over reflections that coincide in $(q_{xy}, q_z)$. β is shown only as powder rings because its contact plane in these films is not established. Spots at $q_z <$ 0.1 Å$^{-1}$ sit on the Yoneda band, where the refraction shift of $q_z$ is largest, so only their $q_{xy}$ is used.

In [ ]:
def g2(X, A, x0, z0, sx, sz, b0, bx, bz):
    x, z = X
    return A * np.exp(-0.5 * (((x - x0) / sx) ** 2 + ((z - z0) / sz) ** 2)) + b0 + bx * (x - x0) + bz * (z - z0)


def fit_spot(img, g, x0, z0, hw=0.10):
    hz = hw if z0 > 0.12 else 0.06
    m = g["ok"] & (np.abs(g["qxy"] - x0) < hw) & (np.abs(g["qz"] - z0) < hz)
    x, z, y = g["qxy"][m], g["qz"][m], img[m]
    p0 = [np.percentile(y, 99) - np.median(y), x0, z0, 0.03, 0.03, np.median(y), 0, 0]
    lb = [0, x0 - hw, z0 - hz, 0.005, 0.005, -np.inf, -np.inf, -np.inf]
    ub = [np.inf, x0 + hw, z0 + hz, 0.2, 0.2, np.inf, np.inf, np.inf]
    P, C = curve_fit(g2, (x, z), y, p0=p0, bounds=(lb, ub), sigma=np.sqrt(np.maximum(y, 1)), maxfev=20000)
    return P, np.sqrt(np.diag(C))


def reflections(cif, qmax=2.2, rel=0.02, normal=(1, 0, 0)):
    st = gemmi.read_small_structure(str(cif))
    st.setup_cell_images()
    sg = gemmi.find_spacegroup_by_name(st.spacegroup_hm) or gemmi.SpaceGroup("P 1")
    ops, calc = sg.operations(), gemmi.StructureFactorCalculatorX(st.cell)
    B = np.array(st.cell.frac.mat.tolist()).T * 2 * np.pi          # rows a*, b*, c*
    n = np.asarray(normal, float) @ B
    n /= np.linalg.norm(n)
    lim = [int(qmax / np.linalg.norm(b)) + 1 for b in B]
    acc = {}
    for h in range(-lim[0], lim[0] + 1):
        for k in range(-lim[1], lim[1] + 1):
            for l in range(-lim[2], lim[2] + 1):
                if (h, k, l) == (0, 0, 0) or ops.is_systematically_absent([h, k, l]):
                    continue
                G = np.array([h, k, l]) @ B
                q = np.linalg.norm(G)
                if q > qmax:
                    continue
                qz = abs(G @ n)
                key = (round(np.sqrt(max(q * q - qz * qz, 0)), 3), round(qz, 3))
                e = acc.setdefault(key, [0.0, []])
                e[0] += abs(calc.calculate_sf_from_small_structure(st, [h, k, l])) ** 2
                e[1].append((h, k, l))
    imax = max(v[0] for v in acc.values())
    return pd.DataFrame([dict(qxy=k[0], qz=k[1], q=np.hypot(*k), I=v[0] / imax, hkl=min(v[1], key=lambda t: (-t[0], abs(t[1]), -t[2])))
                         for k, v in acc.items() if v[0] / imax > rel]).sort_values("q").reset_index(drop=True)


REFL = {lab: reflections(cif) for lab, cif in CELLS.items()}

SPOTS = {"(002)": (0.525, 0.06)}
srows = []
for f, D in DATA.items():
    for lab, (x0, z0) in SPOTS.items():
        for s in (-1, 1):
            if s * x0 > 0.55:
                continue
            try:
                P, E = fit_spot(D["img"], D["g"], s * x0, z0)
            except RuntimeError:
                continue
            if P[0] < 0.5 * P[5] or not (0.008 < P[3] < 0.15 and 0.008 < P[4] < 0.15):   # weak, or not a resolved spot
                continue
            srows.append(dict(frame=f, spot=lab, side=s, qxy=abs(P[1]), qxy_err=E[1], qz=P[2], qz_err=E[2],
                              fwhm_xy=2.355 * P[3], fwhm_z=2.355 * P[4], peak_over_bg=P[0] / P[5]))
SPOT_FITS = pd.DataFrame(srows)
SPOT_MEAN = SPOT_FITS.groupby(["frame", "spot"]).agg(qxy=("qxy", "mean"), qz=("qz", "mean"), n_sides=("side", "size"),
                                                      fwhm_xy=("fwhm_xy", "mean"), fwhm_z=("fwhm_z", "mean")).reset_index()
SPOT_MEAN.round(4)

The in-plane spacing along $c$ follows from (002), $c = 2 \cdot 2\pi/q_{xy}$. The comparison below sets it against the nearest predicted reflection of each α cell, and lists the strongest fiber reflections of each cell near the measured ring and meridian peaks.

In [ ]:
L_INDEX = {"(002)": 2}
cmp_rows = []
for _, r in SPOT_MEAN.iterrows():
    row = dict(frame=r.frame, spot=r.spot, qxy=r.qxy, qz=r.qz, c_from_qxy=L_INDEX[r.spot] * 2 * np.pi / r.qxy)
    for lab in ("α constrained", "α relaxed (MD)"):
        R = REFL[lab]
        w = r.qz > 0.12
        dist = np.hypot(R.qxy - r.qxy, (R.qz - r.qz) * w)
        j = int(np.argmin(dist))
        row[f"{lab} qxy"] = R.qxy[j]
        row[f"{lab} qz"] = R.qz[j]
    cmp_rows.append(row)
INDEX = pd.DataFrame(cmp_rows)
for lab, cif in CELLS.items():
    st = gemmi.read_small_structure(str(cif))
    print(f"{lab:16s} a {st.cell.a:7.3f}  b {st.cell.b:6.3f}  c {st.cell.c:7.3f}  beta {st.cell.beta:6.2f}")
display(INDEX.round(3))
for lab in ("α constrained", "α relaxed (MD)"):
    R_ = REFL[lab]
    print(lab)
    print(R_[(R_.I > 0.05) & (R_.q < 1.3)][["hkl", "qxy", "qz", "q", "I"]].round(3).to_string(index=False))

## Orientation distribution of the lamellar ring

The ring 0.46 < $q$ < 0.56 Å$^{-1}$ is integrated per 2° $\chi$ bin after subtracting the mean of the flanks 0.40 to 0.44 and 0.60 to 0.64 Å$^{-1}$, both $q_{xy}$ signs folded. Bins below $\chi_\mathrm{edge}$ are absent. The ring also carries (002) near $\chi$ = 90°, so the (200) mosaic is fitted on $\chi$ < 20° with a Gaussian core. The excess above the core between 20 and 60° is reported as a fraction of the core integral. With $\chi_\mathrm{edge}$ = 3.3° at this $q$ and 2° bins, a core of $\sigma \approx$ 5° is sampled by only a few bins on its flank and its width is uncertain beyond the fit error.

In [ ]:
CHI_B = np.arange(0, 92, 2.0)
CHI_C = 0.5 * (CHI_B[1:] + CHI_B[:-1])


def chi_profile(img, g, lo, hi):
    sel = g["ok"] & (g["q"] > lo) & (g["q"] < hi)
    s, _ = np.histogram(g["chi"][sel], CHI_B, weights=img[sel])
    n, _ = np.histogram(g["chi"][sel], CHI_B)
    return np.where(n > 3, s / np.maximum(n, 1), np.nan)


MOSAIC = {}
for f, D in DATA.items():
    img, g = D["img"], D["g"]
    net = chi_profile(img, g, 0.46, 0.56) - 0.5 * (chi_profile(img, g, 0.40, 0.44) + chi_profile(img, g, 0.60, 0.64))
    ok = np.isfinite(net) & (CHI_C < 20)
    gf = lambda x, A, s: A * np.exp(-0.5 * (x / s) ** 2)
    P, C = curve_fit(gf, CHI_C[ok], net[ok], p0=[np.nanmax(net), 10])
    resid = net - gf(CHI_C, *P)
    tail = np.isfinite(net) & (CHI_C > 20) & (CHI_C < 60)
    MOSAIC[f] = dict(net=net, P=P, sigma=P[1], sigma_err=np.sqrt(C[1, 1]), fwhm=2.355 * P[1],
                     tail_20_60_over_core=np.nansum(np.clip(resid[tail], 0, None)) / np.nansum(gf(CHI_C, *P)[np.isfinite(net)]),
                     chi_first=CHI_C[np.isfinite(net)][0])
pd.DataFrame({f: {k: v for k, v in M.items() if k not in ("net", "P")} for f, M in MOSAIC.items()}).T.round(3)

## Figure SI 2

(a) Reciprocal-space map, log intensity, with the forbidden wedge left empty and the integration sectors drawn as dashed lines (red, out of plane, $\chi$ < 25°, blue, in plane, 73° < $\chi$ < 90°). Reflection labels as in the original NIKA figure. (b) The two sector cuts on $|q|$, drawn by `giwaxs_style.draw_cut_panel`, which FIG4 uses for SI 12.

In [ ]:
def smooth(y):
    return 10 ** savgol_filter(np.log10(y), 9, 2)


def draw_map(ax, f, labels=True, vmax_scale=3.6):
    D = DATA[f]
    bx, bz, I = qmap(D["img"], D["g"])
    pk = np.nanpercentile(I[(np.abs(bz[:-1, None] - 0.50) < 0.03) & (np.abs(bx[None, :-1]) < 0.08)], 90)
    cm = matplotlib.colormaps["jet"].copy()
    cm.set_bad("white")
    cm.set_over(cm(0.88))
    ax.pcolormesh(bx, bz, np.log10(np.clip(I, 1e-3, None)), cmap=cm, vmin=np.log10(1.4), vmax=np.log10(pk * vmax_scale),
                  shading="flat", rasterized=True)
    ax.set_xlim(bx[0], bx[-1])
    ax.set_ylim(bz[0], bz[-1])
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_xlabel(r"$q_{xy}$ (Å$^{-1}$)")
    ax.set_ylabel(r"$q_z$ (Å$^{-1}$)")
    ax.xaxis.set_major_locator(MultipleLocator(0.5))
    ax.yaxis.set_major_locator(MultipleLocator(0.5))
    ax.tick_params(which="both", color="white")
    glow = [pe.Stroke(linewidth=3.2, foreground="white", alpha=0.45), pe.Normal()]
    R = 3.0
    for sgn in (-1, 1):
        c = np.radians(CHI_OOP)
        ax.plot([0, sgn * R * np.sin(c)], [0, R * np.cos(c)], color="red", lw=1.6, dashes=(4, 2.5), path_effects=glow)
    for cdeg in (CHI_IP, 90.0):
        c = np.radians(cdeg)
        ax.plot([0, -R * np.sin(c)], [0.004, R * np.cos(c) + 0.004], color="#2f6bff", lw=1.6, dashes=(4, 2.5), path_effects=glow)
    return bx, bz, I


# figure cuts on a log grid from 0.1 1/A, the grid of the original NIKA figure; written to SI2_cuts.csv for SI 12
QB_FIG = np.logspace(np.log10(0.1), np.log10(2.5), 181)


def smooth_valid(y):
    """Log-intensity Savitzky-Golay smoothing on the finite positive points, put back on the full grid."""
    out = np.full_like(y, np.nan)
    ok = np.isfinite(y) & (y > 0)
    out[ok] = smooth(y[ok])
    return out


D = DATA[PRIMARY]
FIGCUTS = pd.DataFrame({"q": np.sqrt(QB_FIG[1:] * QB_FIG[:-1])})
for s_ in ("oop", "ip", "meridian"):
    FIGCUTS[s_] = sector(D["img"], D["g"], D["sel"][s_], QB_FIG)[1]
FIGCUTS["oop_smooth"] = smooth_valid(FIGCUTS.oop.to_numpy())
FIGCUTS["ip_smooth"] = smooth_valid(FIGCUTS.ip.to_numpy())
FIGCUTS.to_csv(OUT_DIR / "SI2_cuts.csv", index=False)

fp = PEAKS[PEAKS.frame == PRIMARY].set_index("peak")
CUT_LABELS = {"(200)": ("oop_smooth", fp.q["(200)"]), "(002)": ("ip_smooth", fp.q["IP ring, one peak"], 1.16),
              "(400)": ("oop_smooth", fp.q["(400), one peak"]), r"$\mathbf{(\pi-\pi)}$": ("ip_smooth", fp.q["π-π"], 0.95)}
MAP_LABELS = [("(200)", -0.47, 0.68), ("(400)", 0.39, 1.30), ("(20*)", -0.97, 0.88), ("(002)", -0.55, 0.11),
              (r"$\mathbf{(\pi-\pi)}$", -1.80, 0.11)]

with plt.rc_context(gws.RC):
    fig = plt.figure(figsize=(3.5, 4.6))
    axa = fig.add_axes([0.165, 0.575, 0.81, 0.405])
    axb = fig.add_axes([0.165, 0.085, 0.81, 0.40])
    draw_map(axa, PRIMARY)
    for txt, xx, zz in MAP_LABELS:
        axa.text(xx, zz, txt, **gws.MAP_TEXT)
    gws.draw_cut_panel(axb, FIGCUTS, CUT_LABELS)
    fig.text(0.015, 0.995, "(a)", va="top", fontsize=gws.FS_LABEL + 1)
    fig.text(0.015, 0.515, "(b)", va="top", fontsize=gws.FS_LABEL + 1)
    fig.savefig(OUT_DIR / "SI2.png", dpi=300)
    fig.savefig(OUT_DIR / "SI2.pdf", dpi=300)
    plt.show()

## Figure SI 2, reduction detail

(a) Raw detector image with iso-$|q|$ contours (0.5 Å$^{-1}$ steps, white) and the sector boundaries (red, blue) transformed back to pixels, the direct beam (+), the specular spot (×) and the fitted Yoneda row (dotted). (b) Map with the fitted (002) centres (black crosses) and fiber-texture reflections of the constrained (open circles) and relaxed (open squares) α cells, marker area proportional to $|F|^2$. (c) Meridian and in-plane cuts on a linear $q$ axis with the pseudo-Voigt fits (black), β-ZnPc powder rings (orange ticks, top) and relaxed α rings (green ticks, bottom), each above 10% of its strongest reflection. (d) Background-subtracted lamellar ring against $\chi$, with the Gaussian core fit. The shaded band is $\chi < \chi_\mathrm{edge}$ at $q$ = 0.5 Å$^{-1}$.

In [ ]:
D = DATA[PRIMARY]
img, g = D["img"], D["g"]
G0 = GEOM[PRIMARY]
fig, axs = plt.subplots(2, 2, figsize=(7.2, 6.4))
ax = axs[0, 0]
y0, y1, x0_, x1_ = 520, 1000, 0, 800
sub = img[y0:y1, x0_:x1_]
cm = matplotlib.colormaps["jet"].copy()
cm.set_bad("white")
ax.imshow(np.log10(np.where(sub > 0, sub, np.nan)), cmap=cm, vmin=np.log10(1.4), vmax=np.log10(np.nanpercentile(sub, 99.7)),
          extent=(x0_, x1_, y1, y0), interpolation="nearest")
Yg, Xg = np.mgrid[y0:y1, x0_:x1_]
ax.contour(Xg, Yg, np.where(g["ok"], g["q"], np.nan)[y0:y1, x0_:x1_], levels=[0.5, 1.0, 1.5, 2.0], colors="white", linewidths=0.6)
chi_s = np.where(g["ok"], np.where(g["qxy"] < 0, -1, 1) * g["chi"], np.nan)[y0:y1, x0_:x1_]
ax.contour(Xg, Yg, chi_s, levels=[-CHI_OOP, CHI_OOP], colors="red", linewidths=1.0, linestyles="dashed")
ax.contour(Xg, Yg, chi_s, levels=[-CHI_IP], colors="#2f6bff", linewidths=1.2, linestyles="dashed")
ax.plot(D["cx"], G0["y_direct"], "w+", ms=6)
ax.plot(D["cx"], G0["y_specular"], "wx", ms=4)
ax.axline((D["cx"], G0["y_yoneda"]), slope=np.tan(np.radians(G0["roll_deg"])), color="white", lw=0.8, ls=":")
ax.set_xlim(x0_, x1_)
ax.set_ylim(y1, y0)
ax.set_xlabel("pixel $X$")
ax.set_ylabel("pixel $Y$")
ax.grid(False)

ax = axs[0, 1]
draw_map(ax, PRIMARY, vmax_scale=2.5)
for lab_, mk in (("α constrained", "o"), ("α relaxed (MD)", "s")):
    R = REFL[lab_]
    R = R[R.I > 0.02]
    for sgn in (-1, 1):
        ax.scatter(sgn * R.qxy, R.qz, s=4 + 40 * R.I, marker=mk, facecolors="none", edgecolors="white", linewidths=0.6)
ss = SPOT_FITS[SPOT_FITS.frame == PRIMARY]
ax.plot(ss.side * ss.qxy, ss.qz, "k+", ms=5, mew=0.9)
for t, mk in (("constrained", "o"), ("relaxed", "s")):
    ax.scatter([], [], marker=mk, facecolors="none", edgecolors="0.3", label=t)
ax.legend(loc="upper left", fontsize=6, frameon=True, framealpha=0.8)

ax = axs[1, 0]
for s, c, off in (("meridian", "red", 1.0), ("ip", "blue", 1.0)):
    x, y = CUTS[PRIMARY][s]
    ax.semilogy(x, y, color=c, lw=0.9, label={"meridian": "meridian", "ip": "in plane"}[s])
for (f, lab_), (s, model, (lo, hi), _) in FITS.items():
    if f == PRIMARY and lab_ != "IP ring, one peak":
        xx = np.linspace(lo, hi, 300)
        ax.semilogy(xx, model(xx), "k", lw=0.7)
ylo, yhi = ax.get_ylim()
for lab_, c, (a_, b_) in (("β (CIF, 100 K)", "tab:orange", (0.93, 1.0)), ("α relaxed (MD)", "tab:green", (0.0, 0.07))):
    R = REFL[lab_]
    agg = R.groupby(R.q.round(3)).I.sum()
    agg = agg / agg.max()
    for qq, ii in agg[agg > 0.1].items():
        ax.axvline(qq, ymin=a_, ymax=b_, color=c, lw=0.8)
    ax.plot([], [], color=c, lw=0.8, label="β powder" if lab_.startswith("β") else "α relaxed")
ax.set_xlim(0.3, 2.2)
ax.set_xlabel(r"$q$ (Å$^{-1}$)")
ax.set_ylabel("Intensity (a.u.)")
ax.legend(loc="center right", frameon=False, fontsize=6)

ax = axs[1, 1]
for f, c in ((PRIMARY, "k"), *[(f, "0.55") for f in DATA if f != PRIMARY]):
    M = MOSAIC[f]
    nrm = M["P"][0]
    ax.plot(CHI_C, M["net"] / nrm, "o", ms=2.5, color=c, label=f.replace(".edf", ""))
    xx = np.linspace(0, 40, 300)
    ax.plot(xx, np.exp(-0.5 * (xx / M["P"][1]) ** 2), "-", color=c, lw=0.8)
ax.axvspan(0, chi_edge(0.5, g["k"], g["ai"]), color="0.85", lw=0)
ax.set_xlim(0, 90)
ax.set_yscale("log")
ax.set_ylim(5e-3, 1.5)
ax.set_xlabel(r"$\chi$ (deg)")
ax.set_ylabel("Lamellar ring, normalized")
ax.legend(loc="upper right", fontsize=6, frameon=False)
for a_, t in zip(axs.flat, "abcd"):
    a_.text(-0.2, 1.02, f"({t})", transform=a_.transAxes, va="bottom")
fig.tight_layout()
fig.savefig(OUT_DIR / "SI2_reduction.png", dpi=300)
fig.savefig(OUT_DIR / "SI2_reduction.pdf", dpi=300)
plt.show()

## Export

`SI2_cuts.csv` (written by the figure cell) holds the figure-sector cuts on the figure's log grid, raw and as smoothed for panel (b), and the meridian cut, and `SI2_summary.json` holds under `comparison` the lamellar and π-π spacings that FIG4 (SI 12) sets against the drawn structure.

In [ ]:
summary = dict(
    frames={f: dict(centre=FRAMES[f], **{k: float(v) for k, v in GEOM[f].items()}) for f in DATA},
    alpha_c_film_deg=float(AC),
    q_scale_check=pd.DataFrame(check).to_dict("records"),
    paracrystal={f: {k: float(v) for k, v in p.items()} for f, p in PARA.items()},
    mosaic={f: dict(sigma_deg=float(M["sigma"]), sigma_err=float(M["sigma_err"]), tail_20_60_over_core=float(M["tail_20_60_over_core"]))
            for f, M in MOSAIC.items()},
)
fp = PEAKS[PEAKS.frame == PRIMARY].set_index("peak")
summary["comparison"] = {   # the two spacings FIG4 / SI12 sets against the drawn structure
    name: dict(q=float(fp.q[k]), q_err=float(fp.q_err[k]), d=float(fp.d[k]), d_err=float(fp.d[k] * fp.q_err[k] / fp.q[k]),
               fwhm=float(fp.fwhm[k]), sector=str(fp.sector[k]))
    for name, k in (("d_lam", "(200)"), ("d_pi", "π-π"))
}
PEAKS.to_csv(OUT_DIR / "SI2_peaks.csv", index=False)
SPOT_FITS.to_csv(OUT_DIR / "SI2_spots.csv", index=False)
(OUT_DIR / "SI2_summary.json").write_text(json.dumps(summary, indent=1, default=float))
print("wrote SI2_peaks.csv, SI2_spots.csv, SI2_cuts.csv, SI2_summary.json")

## Findings and open issues (values from the run above)

- The $q$ scale needs no recalibration. With the transmitted-spot centre (584, 982) and the header distance, the meridian and in-plane peak centres agree with the NIKA reduction of the same frame to 0.03 to 0.45%.
- Lamellar spacing is 12.42 Å ($q_{200}$ = 0.506 Å$^{-1}$, vacuum $q$). The relaxed α cell (12.33 Å) is 0.7% short and the constrained cell (12.47 Å) 0.4% long. The refraction correction at the header $\alpha_i$ would lengthen the measured spacing by about 1.8%, which would leave both cells short.
- (002) on the Yoneda band gives $c$ = 23.97 Å, equal to the relaxed α cell (23.99 Å) and 1.5% above the constrained cell (23.62 Å). The single-peak in-plane ring at 0.527 Å$^{-1}$ is a blend of (002) with the in-plane tail of the (200) mosaic.
- π-π is 3.303 Å, against 3.32 Å in the relaxed cell. In fiber texture with $\mathbf{a}^*$ along $q_z$ the α (11$l$) family sits at $q_z \approx$ 0.25 Å$^{-1}$, and the π-π ring here is broad in both $q$ and $\chi$.
- The meridian holds no clean higher orders. The 1.07 Å$^{-1}$ peak is a (400) and (402)/(20$\bar{4}$) blend whose centre depends on the sector, the (400) component from a two-peak fit is 1.02 ± 0.03 Å$^{-1}$ (2.02 $q_1$), and the third-order window is a weak hump. A paracrystal $g$ of 4.5 to 6.5% follows depending on the orders used, so lamellar disorder is not constrained by these data beyond the first-order width (Scherrer length 83 Å, a lower bound).
- The lamellar mosaic is $\sigma$ = 11.9° (FWHM 28°), with excess intensity between 20 and 60° of 13% of the core and a rise toward 90° from (002).
- Open. The specular separation puts $\alpha_i$ at 0.16°, below the film critical angle of 0.185°, which would weight the measurement toward the vacuum side of the film and leaves the refraction correction to $q_z$ undetermined. The β contact plane is not assigned. The NIKA export `XuessGiwaxs.csv` used in FIG4 is from a different, more ordered film (`ZnPc_0_00000`, 2024-04-17) and should be replaced by `XuessGiwaxs_ZnPC_Harlan.csv`.